# Project Policy Challenge -- Simple Qwen Starter

**Model:** `Qwen/Qwen2.5-0.5B-Instruct`  
**Task:** For each question, predict `YES`, `NO`, or `INSUFFICIENT_INFO` using frozen PSU policy passages.

This is deliberately a **naive baseline**, not a strong or sophisticated solution. It does just four things:

1. Find three policy snippets using TF-IDF.
2. Send them and the question to Qwen.
3. Read the first decision label from Qwen's response.
4. Calculate accuracy and macro-F1 on the **24 development questions**.

  
The policy passages are fixed, source-attributed **paraphrases** of official PSU pages, not full webpage copies. THESE ARE CLASSROOM RESEARCH MATERIALS, NOT OFFICIAL ADVISING GUIDANCE.

**For Kaggle:** Import this notebook; use **Add Input** to attach `PSU_Qwen_Simple_Starter_Data.zip`. Kaggle may unpack it into a dataset folder automatically. Enable Internet only if needed to install dependencies / download Qwen.


In [1]:
# 1. Check dependencies (no installation)

import torch
import transformers
import sklearn

from transformers import AutoTokenizer, AutoModelForCausalLM

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Scikit-learn:", sklearn.__version__)

print("Dependencies loaded successfully!")

PyTorch: 2.15.0
Transformers: 5.23.0
Scikit-learn: 1.9.1
Dependencies loaded successfully!


In [2]:
# 2. Load the frozen PSU excerpts, 24 questions, and development labels
from pathlib import Path
import csv, zipfile

filename = 'PSU_24_Questions_v1.csv'
roots = [Path('/kaggle/input'), Path.cwd(), Path('/kaggle/working')]
matches = [p for root in roots if root.exists() for p in root.rglob(filename)
           if (p.parent / 'PSU_24_Dev_Labels_v1.csv').exists()]

# Also works when Kaggle leaves the input ZIP unextracted.
if not matches:
    zips = [p for root in roots if root.exists()
            for p in root.rglob('PSU_Qwen_Simple_Starter_Data.zip')]
    if zips:
        destination = Path('/kaggle/working/psu_simple_data') if Path('/kaggle/working').exists() else Path.cwd()/'psu_simple_data'
        destination.mkdir(exist_ok=True, parents=True)
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(destination)
        matches = list(destination.rglob(filename))
assert matches, 'Attach PSU_Qwen_Simple_Starter_Data.zip using Kaggle Add Input.'
DATA_DIR = matches[0].parent

def read_csv(name):
    with (DATA_DIR/name).open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

passages = read_csv('PSU_Policy_Excerpts_v1.csv')
questions = read_csv('PSU_24_Questions_v1.csv')
gold = {r['id']: r['gold'] for r in read_csv('PSU_24_Dev_Labels_v1.csv')}
print(f'{len(passages)} policy passages; {len(questions)} development questions')
print('Example:', questions[0]['question'])


23 policy passages; 24 development questions
Example: A PSU undergraduate has a term GPA of 2.30 and a PSU cumulative GPA of 2.10. Do both numbers satisfy the published GPA thresholds for good academic standing?


In [3]:
# 3. Very simple retrieval: return the top 3 passages by TF-IDF similarity
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')
docs = [p['heading'] + ' ' + p['text'] for p in passages]
doc_vectors = vectorizer.fit_transform(docs)
TOP_K = 3  # Teams can change this.

def retrieve(question):
    q_vector = vectorizer.transform([question])
    scores = (doc_vectors @ q_vector.T).toarray().ravel()
    indices = scores.argsort()[::-1][:TOP_K]
    return [passages[i] for i in indices]

example_evidence = retrieve(questions[0]['question'])
print('Retrieved IDs:', [p['excerpt_id'] for p in example_evidence])


Retrieved IDs: ['S01-E1', 'S01-E2', 'S02-E2']


In [4]:
# 4. Load Qwen and define a single naive prompting method
import torch, re
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu': torch.set_num_threads(4)
print('Loading on', DEVICE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32
).to(DEVICE).eval()

PROMPT = '''Read the PSU policy excerpts and answer the question using ONLY the excerpts.
YES: The excerpts support the proposition in the question.
NO: The excerpts contradict it.
INSUFFICIENT_INFO: The excerpts do not establish either answer.
Do not treat missing evidence as NO.
Output only one label: YES, NO, or INSUFFICIENT_INFO.'''

def predict(question):
    evidence = retrieve(question)
    context = '\n'.join(f"[{p['excerpt_id']}] {p['text']}" for p in evidence)
    messages = [
        {'role': 'system', 'content': PROMPT},
        {'role': 'user', 'content': f'POLICY:\n{context}\n\nQUESTION: {question}'}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors='pt', return_dict=True, truncation=True, max_length=1800
    ).to(DEVICE)
    with torch.inference_mode():
        output = model.generate(**inputs, do_sample=False, max_new_tokens=20,
                                pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    match = re.match(r'^(?:(?:DECISION|ANSWER)\s*:\s*)?(YES|NO|INSUFFICIENT_INFO)\b', raw, re.I)
    label = match.group(1).upper() if match else 'INVALID'
    return label, raw, [p['excerpt_id'] for p in evidence]

print('Model ready')


Loading on cpu


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
W1120 19:22:11.727000 7590 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model ready


In [5]:
# 5. Run evaluation; start with 3, then change to 24
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter
import pandas as pd

N_QUESTIONS = 24   # Change to 24 for the whole development set.
results = []
for q in questions[:N_QUESTIONS]:
    prediction, raw, ids = predict(q['question'])
    results.append({'id': q['id'], 'question': q['question'],
                    'gold': gold[q['id']], 'prediction': prediction,
                    'retrieved_ids': ';'.join(ids), 'raw_output': raw})
    print(q['id'], 'gold:', gold[q['id']], 'predicted:', prediction)

truth = [r['gold'] for r in results]
predictions = [r['prediction'] for r in results]
print('\nAccuracy:', round(accuracy_score(truth, predictions), 3))
print('Macro-F1:', round(f1_score(truth, predictions,
    labels=['YES', 'NO', 'INSUFFICIENT_INFO'], average='macro', zero_division=0), 3))
print('Predicted labels:', dict(Counter(predictions)))

# Compare with the deliberately trivial always-YES classifier.
print('Always-YES baseline:', round(sum(label == 'YES' for label in truth) / len(truth), 3))

out = Path('/kaggle/working/PSU_Qwen_Starter_Results.csv') if Path('/kaggle/working').exists() else Path('PSU_Qwen_Starter_Results.csv')
pd.DataFrame(results).to_csv(out, index=False)
print('Saved:', out)


Q01 gold: YES predicted: NO
Q02 gold: NO predicted: NO
Q03 gold: INSUFFICIENT_INFO predicted: NO
Q04 gold: YES predicted: NO
Q05 gold: NO predicted: NO
Q06 gold: INSUFFICIENT_INFO predicted: NO
Q07 gold: YES predicted: YES
Q08 gold: NO predicted: NO
Q09 gold: INSUFFICIENT_INFO predicted: NO
Q10 gold: YES predicted: NO
Q11 gold: NO predicted: NO
Q12 gold: INSUFFICIENT_INFO predicted: NO
Q13 gold: YES predicted: NO
Q14 gold: NO predicted: NO
Q15 gold: INSUFFICIENT_INFO predicted: NO
Q16 gold: YES predicted: YES
Q17 gold: NO predicted: NO
Q18 gold: INSUFFICIENT_INFO predicted: NO
Q19 gold: YES predicted: YES
Q20 gold: NO predicted: NO
Q21 gold: INSUFFICIENT_INFO predicted: NO
Q22 gold: YES predicted: YES
Q23 gold: NO predicted: NO
Q24 gold: INSUFFICIENT_INFO predicted: NO

Accuracy: 0.5
Macro-F1: 0.413
Predicted labels: {'NO': 20, 'YES': 4}
Always-YES baseline: 0.333
Saved: PSU_Qwen_Starter_Results.csv


## Starting points for teams

This simple baseline is intentionally imperfect. Teams may improve **retrieval**, **prompting**, **decoding**, **data preparation**, **fine-tuning**, or **evaluation**, but should keep the starting checkpoint fixed and compare on the same development questions.

**Important:** The gold labels are development-only and must never be inserted into a model prompt. The hidden test labels will remain private. Do not interpret these 24 examples as live advice about PSU policy.


# Experiments

Everything above is the unchanged starter notebook (baseline: original prompt + TF-IDF top-3, Accuracy 0.500, Macro-F1 0.413).
The sections below test five hypotheses, each changing one component relative to the baseline.

| Section | Hypothesis | What changes |
|---|---|---|
| Experiment setup | – | device (cuda / mps / cpu) and switchable retrievers shared by all experiments |
| Experiment H1 | a more robust retriever improves evidence recall | retriever: TF-IDF / BM25 / semantic / hybrid |
| Experiment H2 | a structured prompt improves accuracy | prompt |
| Experiment H3 | deciding numeric comparisons by code fixes decimal errors | prompt V2 + code rules |
| Experiment H4 | scoring labels by logits and biasing INSUFFICIENT_INFO corrects the NO bias | decoding (logit scoring + bias) |
| Experiment H5 | LoRA SFT on 1,800 examples improves accuracy | model weights (LoRA adapter) |
| Summary | – | all results in one table |

Unless stated otherwise, every experiment uses **TF-IDF top-3** retrieval (`RETRIEVER = 'tfidf'`, `TOP_K = 3`).

**Extra dependencies** (local): `pip install peft rank_bm25 sentence-transformers` and `pip uninstall -y torchao` (conflicts with peft).
On Kaggle: `%pip install -q rank_bm25` and `%pip install -q -U peft` in a first cell.

## Experiment setup

The starter cells above only check for CUDA and define a single TF-IDF `retrieve(question)`. This cell
(1) selects the device, including Apple Silicon (MPS), and moves the loaded model there, and
(2) replaces `retrieve` with a switchable version: `retrieve(question, method=None, k=None)`.
With the default `RETRIEVER = 'tfidf'` and `TOP_K = 3` it returns the same passages as the starter retriever.

In [6]:
# Experiment setup: device + switchable retrievers (shared by all experiments below)
import re
import numpy as np
import torch
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# ---- Device: cuda / mps / cpu ----
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
model = model.to(DEVICE)        # cell 4 only checks CUDA; move the baseline model if MPS is available
print('Device:', DEVICE)

# ---- TF-IDF (reuses vectorizer / doc_vectors / docs from cell 3) ----
def tfidf_rank(question):
    scores = (doc_vectors @ vectorizer.transform([question]).T).toarray().ravel()
    return list(np.argsort(-scores))

# ---- BM25 ----
_bm25 = None
def tokenize(text):
    return [w for w in re.findall(r'[a-z0-9]+', text.lower()) if w not in ENGLISH_STOP_WORDS]

def bm25_rank(question):
    global _bm25
    if _bm25 is None:                      # build only when first used
        from rank_bm25 import BM25Okapi
        _bm25 = BM25Okapi([tokenize(d) for d in docs])
    return list(np.argsort(-_bm25.get_scores(tokenize(question))))

# ---- Semantic search ----
EMB_ID = 'BAAI/bge-small-en-v1.5'
_embedder, _doc_emb = None, None

def semantic_rank(question):
    global _embedder, _doc_emb
    if _embedder is None:                  # load the embedding model only when first used
        from sentence_transformers import SentenceTransformer
        _embedder = SentenceTransformer(EMB_ID, device=DEVICE)
        _doc_emb = _embedder.encode(docs, normalize_embeddings=True)
    q = _embedder.encode([question], normalize_embeddings=True)[0]
    return list(np.argsort(-(_doc_emb @ q)))

# ---- Hybrid: BM25 + semantic, fused with Reciprocal Rank Fusion ----
def hybrid_rank(question, rrf_k=60):
    score = np.zeros(len(passages))
    for ranking in (bm25_rank(question), semantic_rank(question)):
        for r, i in enumerate(ranking):
            score[i] += 1 / (rrf_k + r + 1)
    return list(np.argsort(-score))

RANKERS = {'tfidf': tfidf_rank, 'bm25': bm25_rank, 'rag': semantic_rank, 'hybrid_rag': hybrid_rank}

# ---- Global config: change these two lines to switch ----
RETRIEVER = 'tfidf'
TOP_K = 3

def retrieve(question, method=None, k=None):
    method = method or RETRIEVER
    k = k or TOP_K
    return [passages[i] for i in RANKERS[method](question)[:k]]

example_evidence = retrieve(questions[0]['question'])
print(f'[{RETRIEVER}, top-{TOP_K}] Retrieved IDs:', [p['excerpt_id'] for p in example_evidence])

Device: mps
[tfidf, top-3] Retrieved IDs: ['S01-E1', 'S01-E2', 'S02-E2']


# Experiment H1: Retrieval

**Hypothesis:** TF-IDF only matches keywords; a more robust retriever will improve evidence recall.

**Change vs. baseline:** retriever only (all retrievers are defined in Experiment setup and switched with `RETRIEVER`). Compared: TF-IDF, BM25,
semantic search (`rag`, `BAAI/bge-small-en-v1.5`), and hybrid (`hybrid_rag`: BM25 + semantic, fused with Reciprocal Rank Fusion, k = 60, equal weights).
Evaluated with Recall@1/2/3 against `evidence_labels.csv`.


In [7]:
# H1.1 Retrieval recall@1/2/3 for any retriever
import csv
import pandas as pd

# read evidence labels
EV_FILE = 'evidence_labels.csv'
ev_path = next(p for root in [DATA_DIR, Path('/kaggle/input'), Path.cwd()]
               if root.exists() for p in root.rglob(EV_FILE))
with ev_path.open(encoding='utf-8-sig', newline='') as f:
    gold_ev = {r['id']: set(r['gold_evidence'].split(';')) for r in csv.DictReader(f)}

KS = [1, 2, 3]

def eval_recall(method, k=None, name=None):
    """
    method: a retriever name in RANKERS (e.g. 'tfidf', 'hybrid'),
            or a function question -> ranked passages (old style).
    k:      how many passages to retrieve (default: max(KS)).
    name:   optional label; built automatically when method is a string.
    """
    k = k or max(KS)
    if isinstance(method, str):
        if method not in RANKERS:
            raise ValueError(f'Unknown retriever {method!r}. Choose from {list(RANKERS)}')
        retrieve_fn = lambda q: retrieve(q, method=method, k=k)
        name = name or method
    else:
        retrieve_fn = method
        name = name or getattr(method, '__name__', 'custom')

    rows = []
    for q in questions:
        gold_ids = gold_ev[q['id']]
        ranked = [p['excerpt_id'] for p in retrieve_fn(q['question'])]
        row = {'id': q['id'],
               'gold_evidence': ';'.join(sorted(gold_ids)),
               'retrieved': ';'.join(ranked[:max(KS)])}
        for kk in KS:
            row[f'recall@{kk}'] = len(gold_ids & set(ranked[:kk])) / len(gold_ids)
        row['missing@3'] = ';'.join(sorted(gold_ids - set(ranked[:3])))
        rows.append(row)
    detail = pd.DataFrame(rows)
    summary = {'retriever': name, **{f'Recall@{kk}': round(detail[f'recall@{kk}'].mean(), 3) for kk in KS}}
    return summary, detail

In [8]:
# H1.2 Compare all retrievers
# Compare all retrievers (table is rebuilt every run, no duplicates)
recall_table = []
for method in RANKERS:
    s, detail = eval_recall(method)
    recall_table.append(s)
    print(f'{method:14s} missing@2:', detail.loc[detail['recall@2'] < 1, 'id'].tolist())
print(pd.DataFrame(recall_table))

tfidf          missing@2: ['Q04', 'Q21', 'Q24']
bm25           missing@2: ['Q04', 'Q21', 'Q24']


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

rag            missing@2: ['Q04', 'Q15', 'Q20']
hybrid_rag     missing@2: ['Q04', 'Q15', 'Q20']
    retriever  Recall@1  Recall@2  Recall@3
0       tfidf     0.792     0.917     1.000
1        bm25     0.792     0.917     0.958
2         rag     0.771     0.917     1.000
3  hybrid_rag     0.792     0.917     1.000


# Experiment H2: Prompt V2

**Hypothesis:** A structured prompt that separates the policy from the student's situation and asks the model
to check each condition will improve accuracy over the baseline prompt.

**Change vs. baseline:** prompt and label parser only (retriever: TF-IDF top-3, model: zero-shot Qwen).


In [9]:
# H2.1 Prompt V2: situation vs policy, quote + check conditions + LABEL
PROMPT_V2 = '''You check whether a student's situation satisfies PSU policy rules.
- <policy> contains the ONLY rules you may use as evidence.
- <situation> contains facts about a student and the question to answer.
Treat these facts as true, but they are NOT evidence about what the policy says.
Steps:
1. Find the policy sentence that governs this situation. Quote it with its [ID].
2. List each condition in that rule and check it against the situation.
3. Decide: YES = the policy rule applies and its conditions are met by the situation.
   NO = the policy explicitly says the opposite, or a stated condition is not met.
   INSUFFICIENT_INFO = no policy sentence addresses this situation.
Never quote the <situation> as evidence.
End with: LABEL: YES, NO, or INSUFFICIENT_INFO'''

def parse_label(raw):
    # prefer the value after "LABEL:" (take the last one)
    m = re.findall(r'LABEL:\s*(INSUFFICIENT_INFO|YES|NO)\b', raw, re.I)
    if m:
        return m[-1].upper()
    # fallback: last label anywhere in the text
    m = re.findall(r'\b(INSUFFICIENT_INFO|YES|NO)\b', raw, re.I)
    return m[-1].upper() if m else 'INVALID'

def predict_v2(question):
    evidence = retrieve(question)
    context = '\n'.join(f"[{p['excerpt_id']}] {p['text']}" for p in evidence)
    messages = [
        {'role': 'system', 'content': PROMPT_V2},
        {'role': 'user', 'content': f'<policy>\n{context}\n</policy>\n\n<situation>\n{question}\n</situation>'}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors='pt', return_dict=True, truncation=True, max_length=1800
    ).to(DEVICE)
    with torch.inference_mode():
        output = model.generate(**inputs, do_sample=False, max_new_tokens=20,
                                pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    return parse_label(raw), raw, [p['excerpt_id'] for p in evidence]

print('Prompt V2 ready')

Prompt V2 ready


In [10]:
# H2.2 Sanity check on one question
label, raw, ids = predict_v2(questions[18]['question'])

print(raw)
print(ids)
print('parsed:', label)

INSUFFICIENT_INFO
['S07-E1', 'S07-E2', 'S06-E1']
parsed: INSUFFICIENT_INFO


In [11]:
# H2.3 Run V2 on the development questions
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
from collections import Counter
import pandas as pd

N_QUESTIONS = 24
LABELS = ['YES', 'NO', 'INSUFFICIENT_INFO']
results_v2 = []
for q in questions[:N_QUESTIONS]:
    prediction, raw, ids = predict_v2(q['question'])
    results_v2.append({'id': q['id'], 'question': q['question'],
                       'gold': gold[q['id']], 'prediction': prediction,
                       'retrieved_ids': ';'.join(ids), 'raw_output': raw})
    print(q['id'], 'gold:', gold[q['id']], 'predicted:', prediction)

truth_v2 = [r['gold'] for r in results_v2]
pred_v2 = [r['prediction'] for r in results_v2]
print('\nV2 Accuracy:', round(accuracy_score(truth_v2, pred_v2), 3))
print('V2 Macro-F1:', round(f1_score(truth_v2, pred_v2, labels=LABELS,
                                     average='macro', zero_division=0), 3))
print('Predicted labels:', dict(Counter(pred_v2)))

Q01 gold: YES predicted: NO
Q02 gold: NO predicted: NO
Q03 gold: INSUFFICIENT_INFO predicted: NO
Q04 gold: YES predicted: INSUFFICIENT_INFO
Q05 gold: NO predicted: NO
Q06 gold: INSUFFICIENT_INFO predicted: NO
Q07 gold: YES predicted: YES
Q08 gold: NO predicted: INSUFFICIENT_INFO
Q09 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q10 gold: YES predicted: INSUFFICIENT_INFO
Q11 gold: NO predicted: NO
Q12 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q13 gold: YES predicted: INSUFFICIENT_INFO
Q14 gold: NO predicted: NO
Q15 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q16 gold: YES predicted: YES
Q17 gold: NO predicted: NO
Q18 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q19 gold: YES predicted: INSUFFICIENT_INFO
Q20 gold: NO predicted: INSUFFICIENT_INFO
Q21 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q22 gold: YES predicted: INSUFFICIENT_INFO
Q23 gold: NO predicted: INSUFFICIENT_INFO
Q24 gold: INSUFFICIENT_INFO predicted: NO

V2 Accuracy: 0.5
V2 Macro-

# Experiment H3: Prompt V2 + numeric comparison by code

**Hypothesis:** Qwen2.5-0.5B makes mistakes when comparing decimals (e.g. Q01: GPA 2.30 vs 2.25).
Deciding numeric threshold questions with code will fix these errors.

**Change vs. H2:** questions with a numeric threshold (and not asking about a future outcome) are decided by code:
thresholds are extracted from the policy with regular expressions and compared with the student's values.
All other questions still go to the V2 model.


In [12]:
# H3.1 V3 = V2 prompt + numeric questions decided by code
import re
import torch

# ======================= Part 1: V2 prompt + model =======================
# PROMPT_V2 and parse_label are defined in Experiment H2 (H2.1).

def model_predict(question, evidence):
    context = '\n'.join(f"[{p['excerpt_id']}] {p['text']}" for p in evidence)
    messages = [
        {'role': 'system', 'content': PROMPT_V2},
        {'role': 'user', 'content': f'<policy>\n{context}\n</policy>\n\n<situation>\n{question}\n</situation>'}
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors='pt', return_dict=True, truncation=True, max_length=1800
    ).to(DEVICE)
    with torch.inference_mode():
        output = model.generate(**inputs, do_sample=False, max_new_tokens=20,
                                pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    return parse_label(raw), raw

# ======================= Part 2: numeric extraction =======================
NUM_WORDS = {w: str(i) for i, w in enumerate(
    'zero one two three four five six seven eight nine ten eleven twelve thirteen '
    'fourteen fifteen sixteen seventeen eighteen nineteen twenty'.split())}
NUM_RE = re.compile(r'(?<![\w.])\$?(\d+(?:\.\d+)?)(?![\w])%?')      # skips IDs like S01-E1
PRE_OPS = [   # comparison words before a number (longer phrases first)
    (r'greater than or equal to|no less than|at least|a minimum of|minimum of|>=|≥', '>='),
    (r'less than or equal to|no more than|at most|a maximum of|maximum of|up to|<=|≤', '<='),
    (r'more than|greater than|higher than|above|exceeds?|exceeding|>', '>'),
    (r'less than|lower than|fewer than|below|under|<', '<'),
]
POST_OPS = [  # comparison words after a number, e.g. "2.00 or higher"
    (r'or (?:higher|more|above|better|greater)', '>='),
    (r'or (?:lower|less|below|fewer)', '<='),
]
UNITS = ['gpa', 'credit', 'hour', 'day', 'week', 'month', 'year', 'percent', '%']
MODIFIERS = {'cumulative': 'cumulative', 'overall': 'cumulative', 'term': 'term', 'semester': 'term'}
OPS = {'>=': lambda a, b: a >= b, '>': lambda a, b: a > b,
       '<=': lambda a, b: a <= b, '<': lambda a, b: a < b}

def normalize_numbers(text):
    """'six credits' -> '6 credits'"""
    return re.sub(r'\b(' + '|'.join(NUM_WORDS) + r')\b',
                  lambda m: NUM_WORDS[m.group(1).lower()], text, flags=re.I)

def _nearest(before, after, words):
    """keyword closest to the number (text before the number is preferred)"""
    best, dist = None, 10**9
    for w in words:
        for m in re.finditer(re.escape(w), before):
            d = len(before) - m.end()
            if d < dist:
                best, dist = w, d
        m = re.search(re.escape(w), after)
        if m and m.start() + 5 < dist:
            best, dist = w, m.start() + 5
    return best

def tag_number(text, s, e):
    """unit (gpa/credit/...) and modifier (term/cumulative) of a number"""
    before, after = text[max(0, s - 50):s].lower(), text[e:e + 30].lower()
    unit = _nearest(before, after, UNITS)
    mod = MODIFIERS.get(_nearest(before[-45:], after[:15], list(MODIFIERS)))
    return unit, mod

def find_op(text, s, e):
    """comparison operator attached to a policy number, or None if it is not a threshold"""
    before, after = text[max(0, s - 40):s].lower(), text[e:e + 25].lower()
    for pat, op in POST_OPS:
        if re.match(r'\s*(?:' + pat + r')', after):
            return op
    for pat, op in PRE_OPS:
        if re.search(r'(?:' + pat + r')\s*(?:a |an |the |of |gpa |cumulative |term )*\$?$', before):
            return op
    return None

def extract_thresholds(passage):
    text, out = normalize_numbers(passage['text']), []
    for m in NUM_RE.finditer(text):
        op = find_op(text, m.start(), m.end())
        if op:
            unit, mod = tag_number(text, m.start(), m.end())
            out.append((unit, mod, op, float(m.group(1))))
    return out

def extract_student_values(question, evidence):
    text = normalize_numbers(question)
    policy_nums = {float(m.group(1)) for p in evidence
                   for m in NUM_RE.finditer(normalize_numbers(p['text']))}
    out = []
    for m in NUM_RE.finditer(text):
        v = float(m.group(1))
        if v in policy_nums:                       # restated from the policy, not a student value
            continue
        if v.is_integer() and 1900 <= v <= 2100:   # skip years
            continue
        unit, mod = tag_number(text, m.start(), m.end())
        if unit:
            out.append((unit, mod, v))
    return out

def numeric_comparisons(question, evidence):
    """list of (excerpt_id, description, passed)"""
    out = []
    for q_unit, q_mod, q_val in extract_student_values(question, evidence):
        for p in evidence:
            for t_unit, t_mod, op, t_val in extract_thresholds(p):
                if q_unit != t_unit or (q_mod and t_mod and q_mod != t_mod):
                    continue                       # same unit only; never term vs cumulative
                out.append((p['excerpt_id'],
                            f'{q_mod or ""} {q_unit} {q_val:g} {op} {t_val:g}'.strip(),
                            OPS[op](q_val, t_val)))
    return out

# ======================= Part 3: decision =======================
PREDICTION_Q = re.compile(r'\bwill\b|\bdefinitely\b|\bguarantee', re.I)   # asks about a future outcome

def predict_v3(question):
    evidence = retrieve(question)
    ids = [p['excerpt_id'] for p in evidence]
    comps = numeric_comparisons(question, evidence)

    # numeric threshold question -> decided by code
    if comps and not PREDICTION_Q.search(question):
        top_id = next(p['excerpt_id'] for p in evidence if any(c[0] == p['excerpt_id'] for c in comps))
        used = [c for c in comps if c[0] == top_id]
        label = 'YES' if all(ok for _, _, ok in used) else 'NO'
        raw = f'[CODE] {top_id}: ' + '; '.join(f'{d} -> {"TRUE" if ok else "FALSE"}' for _, d, ok in used)
        return label, raw, ids

    # everything else -> V2 model
    label, raw = model_predict(question, evidence)
    return label, raw, ids

# check which dev questions are decided by code
for q in questions:
    ev = retrieve(q['question'])
    if numeric_comparisons(q['question'], ev) and not PREDICTION_Q.search(q['question']):
        print(q['id'], '->', predict_v3(q['question'])[:2])
print('\nV3 ready')

Q01 -> ('YES', '[CODE] S01-E1: term gpa 2.3 >= 2.25 -> TRUE; cumulative gpa 2.1 >= 2 -> TRUE')
Q02 -> ('NO', '[CODE] S01-E1: term gpa 2.3 >= 2.25 -> TRUE; cumulative gpa 1.9 >= 2 -> FALSE')
Q10 -> ('YES', '[CODE] S04-E1: credit 18 <= 45 -> TRUE; credit 18 <= 20 -> TRUE')
Q11 -> ('NO', '[CODE] S04-E1: credit 23 <= 45 -> TRUE; credit 23 <= 20 -> FALSE; credit 23 <= 45 -> TRUE; credit 23 <= 20 -> FALSE')

V3 ready


In [13]:
# H3.2 Run V3 on the development questions
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score
from collections import Counter
import pandas as pd

N_QUESTIONS = 24
LABELS = ['YES', 'NO', 'INSUFFICIENT_INFO']
results_v3 = []
for q in questions[:N_QUESTIONS]:
    prediction, raw, ids = predict_v3(q['question'])
    results_v3.append({'id': q['id'], 'question': q['question'],
                       'gold': gold[q['id']], 'prediction': prediction,
                       'retrieved_ids': ';'.join(ids), 'raw_output': raw})
    print(q['id'], 'gold:', gold[q['id']], 'predicted:', prediction)

truth_v3 = [r['gold'] for r in results_v3]
pred_v3 = [r['prediction'] for r in results_v3]
print('\nV3 Accuracy:', round(accuracy_score(truth_v3, pred_v3), 3))
print('V3 Macro-F1:', round(f1_score(truth_v3, pred_v3, labels=LABELS,
                                     average='macro', zero_division=0), 3))
print('Predicted labels:', dict(Counter(pred_v3)))

Q01 gold: YES predicted: YES
Q02 gold: NO predicted: NO
Q03 gold: INSUFFICIENT_INFO predicted: NO
Q04 gold: YES predicted: INSUFFICIENT_INFO
Q05 gold: NO predicted: NO
Q06 gold: INSUFFICIENT_INFO predicted: NO
Q07 gold: YES predicted: YES
Q08 gold: NO predicted: INSUFFICIENT_INFO
Q09 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q10 gold: YES predicted: YES
Q11 gold: NO predicted: NO
Q12 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q13 gold: YES predicted: INSUFFICIENT_INFO
Q14 gold: NO predicted: NO
Q15 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q16 gold: YES predicted: YES
Q17 gold: NO predicted: NO
Q18 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q19 gold: YES predicted: INSUFFICIENT_INFO
Q20 gold: NO predicted: INSUFFICIENT_INFO
Q21 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q22 gold: YES predicted: INSUFFICIENT_INFO
Q23 gold: NO predicted: INSUFFICIENT_INFO
Q24 gold: INSUFFICIENT_INFO predicted: NO

V3 Accuracy: 0.583
V3 Macro-F1: 0.597
P

# Experiment H4: Logit scoring + bias toward INSUFFICIENT_INFO

**Hypothesis:** The baseline model strongly favours NO and never predicts INSUFFICIENT_INFO.
Scoring the three labels directly from the next-token logits, and adding a bias to INSUFFICIENT_INFO,
will correct this and improve accuracy.

**Change vs. baseline:** decoding only (same prompt, retriever TF-IDF top-3, zero-shot model).
Instead of generating text, we take the log-probability of the first token of each label, add a bias
`b_ins` to INSUFFICIENT_INFO, and pick the highest. The bias is swept from 0 to 8 in steps of 0.5;
the best value on the dev set (3.5) is used for the final run, so it is tuned on the dev set.

These cells must run before H5.3, which deletes the baseline `model`.

In [14]:
# H4.1 Check the label tokens: the first token of each label must be different
import numpy as np
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score

LABELS = ['YES', 'NO', 'INSUFFICIENT_INFO']

for l in LABELS:
    ids = tokenizer.encode(l, add_special_tokens=False)
    print(l, ids, [tokenizer.decode([i]) for i in ids])

first_ids = [tokenizer.encode(l, add_special_tokens=False)[0] for l in LABELS]
assert len(set(first_ids)) == 3, 'first tokens collide; rename the labels (e.g. A/B/C)'

YES [14004] ['YES']
NO [8996] ['NO']
INSUFFICIENT_INFO [9557, 8825, 89625, 9068] ['INS', 'UFF', 'ICIENT', '_INFO']


In [15]:
# H4.2 Score the three labels from the next-token logits (baseline prompt)
def label_scores(question):
    evidence = retrieve(question)
    context = '\n'.join(f"[{p['excerpt_id']}] {p['text']}" for p in evidence)
    messages = [
        {'role': 'system', 'content': PROMPT},
        {'role': 'user', 'content': f'POLICY:\n{context}\n\nQUESTION: {question}'},
    ]
    # No truncation here: the original truncation=True could cut the question
    # and generation prompt off the END of a long prompt. We check length instead.
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors='pt', return_dict=True,
    ).to(DEVICE)
    with torch.inference_mode():
        logits = model(**inputs).logits[0, -1].float()   # next-token logits
    logp = torch.log_softmax(logits, dim=-1)              # over the FULL vocab
    return {
        'logp': logp[first_ids].cpu().numpy(),            # log-prob of each label's first token
        'mass': float(logp[first_ids].exp().sum()),       # prob mass on the 3 labels
        'top_token': tokenizer.decode([int(logp.argmax())]),
        'n_tokens': inputs['input_ids'].shape[1],
        'ids': [p['excerpt_id'] for p in evidence],
    }

cache = {q['id']: label_scores(q['question']) for q in questions}
print('Scored', len(cache), 'questions')

Scored 24 questions


In [16]:
# H4.3 Sanity check: does the model put its probability on the three labels?
print('max prompt tokens:', max(c['n_tokens'] for c in cache.values()))
print('avg prob mass on the 3 labels:', round(np.mean([c['mass'] for c in cache.values()]), 3))
print('most common top tokens:', Counter(c['top_token'] for c in cache.values()).most_common(5))
# If mass is low or top tokens aren't YES/NO/INS..., the model wants to start with something
# else (e.g. "The"); fix the prompt before trusting any of the numbers below.

max prompt tokens: 294
avg prob mass on the 3 labels: 0.994
most common top tokens: [('NO', 20), ('YES', 4)]


In [17]:
# H4.4 Sweep the bias added to INSUFFICIENT_INFO
def decide(logp, b_ins=0.0, b_no=0.0):
    s = logp.copy()
    s[1] += b_no
    s[2] += b_ins
    return LABELS[int(s.argmax())]

truth_logit = [gold[q['id']] for q in questions]
print('bias  acc   macroF1  predictions')
for b in np.arange(0, 8.5, 0.5):
    preds = [decide(cache[q['id']]['logp'], b_ins=b) for q in questions]
    acc = accuracy_score(truth_logit, preds)
    f1 = f1_score(truth_logit, preds, labels=LABELS, average='macro', zero_division=0)
    print(f'{b:4.1f}  {acc:.3f}  {f1:.3f}   {dict(Counter(preds))}')

bias  acc   macroF1  predictions
 0.0  0.500  0.413   {'NO': 20, 'YES': 4}
 0.5  0.500  0.413   {'NO': 20, 'YES': 4}
 1.0  0.500  0.413   {'NO': 20, 'YES': 4}
 1.5  0.500  0.413   {'NO': 20, 'YES': 4}
 2.0  0.500  0.413   {'NO': 20, 'YES': 4}
 2.5  0.500  0.413   {'NO': 20, 'YES': 4}
 3.0  0.500  0.413   {'NO': 20, 'YES': 4}
 3.5  0.542  0.550   {'NO': 15, 'INSUFFICIENT_INFO': 5, 'YES': 4}
 4.0  0.375  0.378   {'INSUFFICIENT_INFO': 13, 'NO': 9, 'YES': 2}
 4.5  0.417  0.367   {'INSUFFICIENT_INFO': 19, 'YES': 2, 'NO': 3}
 5.0  0.417  0.311   {'INSUFFICIENT_INFO': 22, 'YES': 2}
 5.5  0.375  0.246   {'INSUFFICIENT_INFO': 23, 'YES': 1}
 6.0  0.333  0.167   {'INSUFFICIENT_INFO': 24}
 6.5  0.333  0.167   {'INSUFFICIENT_INFO': 24}
 7.0  0.333  0.167   {'INSUFFICIENT_INFO': 24}
 7.5  0.333  0.167   {'INSUFFICIENT_INFO': 24}
 8.0  0.333  0.167   {'INSUFFICIENT_INFO': 24}


In [18]:
# H4.5 How far is INSUFFICIENT_INFO from winning on the INSUFFICIENT_INFO questions?
for q in questions:
    if gold[q['id']] == 'INSUFFICIENT_INFO':
        s = cache[q['id']]['logp']
        p = np.exp(s - s.max()); p /= p.sum()             # softmax over just the 3 labels
        rank = int((s > s[2]).sum()) + 1                  # 1 = already winning
        gap = float(s.max() - s[2])                       # bias needed to make it win
        print(q['id'], f'rank={rank}', f'gap={gap:.2f}',
              dict(zip(LABELS, np.round(p, 2))))

B_INS = 3.5   # best bias from the sweep in H4.4 (chosen on the dev set)

def predict_logit(question, b_ins=B_INS):
    out = label_scores(question)
    label = decide(out['logp'], b_ins=b_ins)
    raw = 'logp(YES,NO,INS)=' + ','.join(f'{x:.2f}' for x in out['logp'])
    return label, raw, out['ids']

Q03 rank=3 gap=3.31 {'YES': np.float32(0.28), 'NO': np.float32(0.7), 'INSUFFICIENT_INFO': np.float32(0.03)}
Q06 rank=3 gap=4.08 {'YES': np.float32(0.18), 'NO': np.float32(0.81), 'INSUFFICIENT_INFO': np.float32(0.01)}
Q09 rank=3 gap=4.36 {'YES': np.float32(0.11), 'NO': np.float32(0.88), 'INSUFFICIENT_INFO': np.float32(0.01)}
Q12 rank=3 gap=4.28 {'YES': np.float32(0.38), 'NO': np.float32(0.61), 'INSUFFICIENT_INFO': np.float32(0.01)}
Q15 rank=3 gap=3.46 {'YES': np.float32(0.05), 'NO': np.float32(0.92), 'INSUFFICIENT_INFO': np.float32(0.03)}
Q18 rank=3 gap=3.92 {'YES': np.float32(0.15), 'NO': np.float32(0.84), 'INSUFFICIENT_INFO': np.float32(0.02)}
Q21 rank=3 gap=3.35 {'YES': np.float32(0.15), 'NO': np.float32(0.83), 'INSUFFICIENT_INFO': np.float32(0.03)}
Q24 rank=3 gap=4.58 {'YES': np.float32(0.05), 'NO': np.float32(0.94), 'INSUFFICIENT_INFO': np.float32(0.01)}


In [19]:
# H4.6 Run logit scoring + bias on the development questions
import pandas as pd

results_logit = []
for q in questions:
    prediction, raw, ids = predict_logit(q['question'])
    results_logit.append({'id': q['id'], 'question': q['question'],
                          'gold': gold[q['id']], 'prediction': prediction,
                          'retrieved_ids': ';'.join(ids), 'raw_output': raw})
    print(q['id'], 'gold:', gold[q['id']], 'predicted:', prediction)

truth_l = [r['gold'] for r in results_logit]
pred_l = [r['prediction'] for r in results_logit]
print(f'\nb_ins = {B_INS}')
print('Accuracy:', round(accuracy_score(truth_l, pred_l), 3))
print('Macro-F1:', round(f1_score(truth_l, pred_l, labels=LABELS, average='macro', zero_division=0), 3))
print('Predicted labels:', dict(Counter(pred_l)))

out = Path('/kaggle/working/PSU_Qwen_Logit_Results.csv') if Path('/kaggle/working').exists() else Path('PSU_Qwen_Logit_Results.csv')
pd.DataFrame(results_logit).to_csv(out, index=False)
print('Saved:', out)

Q01 gold: YES predicted: NO
Q02 gold: NO predicted: NO
Q03 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q04 gold: YES predicted: NO
Q05 gold: NO predicted: NO
Q06 gold: INSUFFICIENT_INFO predicted: NO
Q07 gold: YES predicted: YES
Q08 gold: NO predicted: NO
Q09 gold: INSUFFICIENT_INFO predicted: NO
Q10 gold: YES predicted: NO
Q11 gold: NO predicted: INSUFFICIENT_INFO
Q12 gold: INSUFFICIENT_INFO predicted: NO
Q13 gold: YES predicted: NO
Q14 gold: NO predicted: INSUFFICIENT_INFO
Q15 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q16 gold: YES predicted: YES
Q17 gold: NO predicted: NO
Q18 gold: INSUFFICIENT_INFO predicted: NO
Q19 gold: YES predicted: YES
Q20 gold: NO predicted: NO
Q21 gold: INSUFFICIENT_INFO predicted: INSUFFICIENT_INFO
Q22 gold: YES predicted: YES
Q23 gold: NO predicted: NO
Q24 gold: INSUFFICIENT_INFO predicted: NO

b_ins = 3.5
Accuracy: 0.542
Macro-F1: 0.55
Predicted labels: {'NO': 15, 'INSUFFICIENT_INFO': 5, 'YES': 4}
Saved: PSU_Qwen_Logit_Results.csv


# Experiment H5: LoRA supervised fine-tuning (SFT)

**Hypothesis:** Fine-tuning Qwen with LoRA on the provided 1,800 labelled training examples will improve accuracy far more than prompting.

**Change vs. baseline:** model weights (LoRA adapter). Retriever: TF-IDF top-3. Prompt: a concise label-only prompt,
identical at training and inference time (the training data has labels but no rationales, so the step-by-step V2 prompt is not used).

**Training data:** `sft_train_1800.jsonl`: 1,500 fictional-college examples + 300 extra PSU-policy examples; 600 per label.
Each example has a question, three passages (one relevant + distractors) and a label. No development question or development passage appears in it.

**Setup:** LoRA r = 16, alpha = 32, dropout 0.05 on all attention and MLP projections; lr 2e-4, 2 epochs, effective batch 8 (450 steps);
loss only on the answer tokens; seed 0.

In [20]:
# H5.1 Load the training file and define the SFT prompt
import json, math, random, re, time
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

LABELS = ['YES', 'NO', 'INSUFFICIENT_INFO']
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()

TRAIN_FILE = 'sft_train_1800.jsonl'
train_path = next(p for root in [Path('/kaggle/input'), Path.cwd(), WORK] if root.exists()
                  for p in root.rglob(TRAIN_FILE))
with train_path.open(encoding='utf-8') as f:
    train_all = [json.loads(line) for line in f]
print(len(train_all), 'training examples from', train_path)
print('labels :', Counter(x['label'] for x in train_all))
print('sources:', Counter(x['source'] for x in train_all))

# Same prompt is used for training and inference.
SFT_SYSTEM_PROMPT = '''Read the policy excerpts and answer the question using ONLY the excerpts.
YES: the excerpts support the proposition in the question.
NO: the excerpts contradict it.
INSUFFICIENT_INFO: the excerpts do not establish either answer (for example, the question asks
what a specific person will decide, an exact personal outcome, or something the excerpts do not cover).
Output only one label: YES, NO, or INSUFFICIENT_INFO.'''

def sft_user_message(question, evidence):
    context = '\n'.join(f"[{p['excerpt_id']}] {p['text']}" for p in evidence)
    return f'POLICY:\n{context}\n\nQUESTION: {question}'

def sft_chat_prompt(question, evidence):
    msgs = [{'role': 'system', 'content': SFT_SYSTEM_PROMPT},
            {'role': 'user', 'content': sft_user_message(question, evidence)}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

print('\nExample training prompt:\n' + sft_user_message(train_all[0]['question'], train_all[0]['passages']),
      '\n=>', train_all[0]['label'])

1800 training examples from /Users/yangxingchi/LLM/psu_simple_data/PSU_Qwen_Simple_Starter_Data/sft_train_1800.jsonl
labels : Counter({'NO': 600, 'YES': 600, 'INSUFFICIENT_INFO': 600})
sources: Counter({'fictional': 1500, 'psu_extra': 300})

Example training prompt:
POLICY:
[F15-E1] Ashford College gives housing priority to first-year students and graduate students. Seniors, non-degree students, sophomores, and juniors are housed only if space remains after priority groups are placed.
[F16-E4] Northvale University guidance: Dropping below half-time enrollment may affect a student’s scholarship eligibility.
[F20-E3] Harborview University: For a full lab section, a student may still be added with the lab coordinator’s approval on an Add Authorization Form.

QUESTION: Under Ashford College’s policy, can sophomores receive on-campus housing priority? 
=> NO


In [21]:
# H5.2 Training / inference / evaluation functions
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def sft_encode(ex, max_len=768):
    p_ids = tokenizer(sft_chat_prompt(ex['question'], ex['passages']), add_special_tokens=False)['input_ids']
    t_ids = tokenizer(ex['label'] + '<|im_end|>', add_special_tokens=False)['input_ids']
    p_ids = p_ids[-(max_len - len(t_ids)):]
    return {'input_ids': p_ids + t_ids, 'labels': [-100] * len(p_ids) + t_ids}   # loss only on the answer

def sft_collate(batch):
    L = max(len(b['input_ids']) for b in batch)
    pad = tokenizer.pad_token_id
    ids = torch.tensor([b['input_ids'] + [pad] * (L - len(b['input_ids'])) for b in batch])
    lab = torch.tensor([b['labels'] + [-100] * (L - len(b['labels'])) for b in batch])
    att = torch.tensor([[1] * len(b['input_ids']) + [0] * (L - len(b['input_ids'])) for b in batch])
    return ids, att, lab

def load_base_fp32():
    # fp32 master weights for stable LoRA training; fp16 autocast for speed (T4 has no bf16)
    return AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32).to(DEVICE)

def train_lora(train_data, seed=0, epochs=2, lr=2e-4, batch_size=4, grad_accum=2, r=16, alpha=32, log_every=25):
    """LoRA SFT. Memory-saving: logits are computed only at answer positions (Qwen's vocab is 152k,
    full logits for every prompt token would need several GB). Effective batch = batch_size * grad_accum."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    m = get_peft_model(load_base_fp32(), LoraConfig(
        r=r, lora_alpha=alpha, lora_dropout=0.05, task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']))
    m.print_trainable_parameters()
    base = m.get_base_model()          # LoRA layers are already injected inside it
    enc = [sft_encode(x) for x in train_data]
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=lr, weight_decay=0.0)
    n_batches = math.ceil(len(enc) / batch_size)
    steps = epochs * math.ceil(n_batches / grad_accum)
    warm = max(1, int(0.05 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler(enabled=DEVICE == 'cuda')
    m.train(); step, t0, losses = 0, time.time(), []
    for ep in range(epochs):
        order = list(range(len(enc))); random.shuffle(order)
        batches = [order[i:i + batch_size] for i in range(0, len(order), batch_size)]
        for b_idx, b in enumerate(batches):
            ids, att, lab = (t.to(DEVICE) for t in sft_collate([enc[j] for j in b]))
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
                hidden = base.model(input_ids=ids, attention_mask=att).last_hidden_state
                tgt = lab[:, 1:]
                keep = tgt != -100                                  # next-token targets inside the answer
                logits = base.lm_head(hidden[:, :-1][keep])
            loss = torch.nn.functional.cross_entropy(logits.float(), tgt[keep])
            scaler.scale(loss / grad_accum).backward()
            losses.append(loss.item())
            if (b_idx + 1) % grad_accum == 0 or b_idx + 1 == len(batches):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
                step += 1
                if step % log_every == 0 or step == steps:
                    print(f'epoch {ep + 1} step {step}/{steps} loss {np.mean(losses[-log_every * grad_accum:]):.4f} '
                          f'({time.time() - t0:.0f}s)')
    m.eval()
    return m

def sft_parse(raw):
    hit = re.search(r'\b(INSUFFICIENT_INFO|YES|NO)\b', raw.upper())
    return hit.group(1) if hit else 'INVALID'

@torch.inference_mode()
def sft_predict(m, question):
    evidence = retrieve(question)
    enc = tokenizer(sft_chat_prompt(question, evidence), return_tensors='pt', add_special_tokens=False).to(DEVICE)
    with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=DEVICE == 'cuda'):
        out = m.generate(**enc, do_sample=False, max_new_tokens=8, pad_token_id=tokenizer.pad_token_id)
    raw = tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    return sft_parse(raw), raw, [p['excerpt_id'] for p in evidence]

def sft_evaluate(m, name, verbose=False):
    m.eval(); rows = []
    for q in questions:
        pred, raw, ids = sft_predict(m, q['question'])
        rows.append(dict(id=q['id'], question=q['question'], gold=gold[q['id']], prediction=pred,
                         evidence_ids=ids[0], retrieved_ids=';'.join(ids), raw_output=raw))
        if verbose:
            print(q['id'], 'gold:', gold[q['id']], 'pred:', pred)
    df = pd.DataFrame(rows)
    res = dict(system=name,
               accuracy=round(accuracy_score(df.gold, df.prediction), 3),
               macro_f1=round(f1_score(df.gold, df.prediction, labels=LABELS, average='macro', zero_division=0), 3),
               pred_counts=dict(Counter(df.prediction)))
    return res, df

print('SFT functions ready')

SFT functions ready


### H5.3 Train and save the LoRA adapter

Trains LoRA on all 1,800 training examples and saves the adapter to `qwen_policy_lora/` (about 20 minutes).

**Already trained?** Skip this cell and go straight to H5.4, which loads the saved adapter
(from the notebook folder, `/kaggle/working`, or a dataset attached with **Add Input**).

In [22]:
# H5.3 Train on all 1800 examples and save the adapter
# (takes ~20 min; to skip training, run H5.4 instead to load the saved adapter)
import gc
if 'model' in globals():
    del model                      # the fp16 baseline model from the starter cells is no longer needed
for _name in ['sft_model', 'frozen']:
    globals().pop(_name, None)
gc.collect(); torch.cuda.empty_cache()
SEED = 0
sft_model = train_lora(train_all, seed=SEED)

ADAPTER_DIR = WORK / 'qwen_policy_lora'
sft_model.save_pretrained(ADAPTER_DIR)
print('saved adapter ->', ADAPTER_DIR)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497
epoch 1 step 25/450 loss 0.3170 (61s)
epoch 1 step 50/450 loss 0.2320 (116s)
epoch 1 step 75/450 loss 0.2781 (170s)
epoch 1 step 100/450 loss 0.2389 (225s)
epoch 1 step 125/450 loss 0.2200 (283s)
epoch 1 step 150/450 loss 0.1511 (342s)
epoch 1 step 175/450 loss 0.0865 (401s)
epoch 1 step 200/450 loss 0.1528 (461s)
epoch 1 step 225/450 loss 0.1453 (516s)
epoch 2 step 250/450 loss 0.0638 (573s)
epoch 2 step 275/450 loss 0.0573 (629s)
epoch 2 step 300/450 loss 0.0458 (683s)
epoch 2 step 325/450 loss 0.0687 (744s)
epoch 2 step 350/450 loss 0.0737 (802s)
epoch 2 step 375/450 loss 0.0311 (860s)
epoch 2 step 400/450 loss 0.0523 (914s)
epoch 2 step 425/450 loss 0.0432 (973s)
epoch 2 step 450/450 loss 0.0534 (1031s)
saved adapter -> /Users/yangxingchi/LLM/qwen_policy_lora


### H5.4 Load the saved adapter (reproduce without training)

To reproduce the H5 results without retraining: run all cells up to and including H5.2, **skip H5.3**,
then run H5.4 and H5.5. H5.4 loads the trained LoRA adapter (`qwen_policy_lora/`) from `/kaggle/input`
(attached with **Add Input**), `/kaggle/working`, or the notebook folder; H5.5 evaluates it on the 24 development questions.

In [23]:
# H5.4 Load the saved LoRA adapter
import gc
from peft import PeftModel

adapter_dir = next((p.parent for root in [Path('/kaggle/input'), WORK, Path.cwd()] if root.exists()
                    for p in root.rglob('adapter_config.json')), None)
assert adapter_dir is not None, 'Adapter not found: attach qwen_policy_lora with Add Input, or run H5.3 first.'
print('Loading adapter from', adapter_dir)

globals().pop('sft_model', None); gc.collect()
sft_model = PeftModel.from_pretrained(load_base_fp32(), adapter_dir).eval()
print('Adapter loaded')

Loading adapter from /Users/yangxingchi/LLM/qwen_policy_lora


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Adapter loaded


In [24]:
# H5.5 Evaluate on the 24 dev questions and save predictions
main_res, main_df = sft_evaluate(sft_model, 'LoRA SFT', verbose=True)
print(main_res)
print(pd.DataFrame(confusion_matrix(main_df.gold, main_df.prediction, labels=LABELS),
                   index=['gold ' + l for l in LABELS], columns=['pred ' + l for l in LABELS]))

main_df.to_csv(WORK / 'dev_predictions_sft.csv', index=False)
print('predictions -> dev_predictions_sft.csv')

Q01 gold: YES pred: YES
Q02 gold: NO pred: NO
Q03 gold: INSUFFICIENT_INFO pred: INSUFFICIENT_INFO
Q04 gold: YES pred: NO
Q05 gold: NO pred: YES
Q06 gold: INSUFFICIENT_INFO pred: INSUFFICIENT_INFO
Q07 gold: YES pred: YES
Q08 gold: NO pred: NO
Q09 gold: INSUFFICIENT_INFO pred: INSUFFICIENT_INFO
Q10 gold: YES pred: YES
Q11 gold: NO pred: NO
Q12 gold: INSUFFICIENT_INFO pred: INSUFFICIENT_INFO
Q13 gold: YES pred: YES
Q14 gold: NO pred: NO
Q15 gold: INSUFFICIENT_INFO pred: INSUFFICIENT_INFO
Q16 gold: YES pred: YES
Q17 gold: NO pred: NO
Q18 gold: INSUFFICIENT_INFO pred: INSUFFICIENT_INFO
Q19 gold: YES pred: YES
Q20 gold: NO pred: NO
Q21 gold: INSUFFICIENT_INFO pred: INSUFFICIENT_INFO
Q22 gold: YES pred: YES
Q23 gold: NO pred: NO
Q24 gold: INSUFFICIENT_INFO pred: INSUFFICIENT_INFO
{'system': 'LoRA SFT', 'accuracy': 0.917, 'macro_f1': 0.917, 'pred_counts': {'YES': 8, 'NO': 8, 'INSUFFICIENT_INFO': 8}}
                        pred YES  pred NO  pred INSUFFICIENT_INFO
gold YES                     

# Summary

Collects the results of every experiment above into one table (no model is re-run).

In [25]:
# Summary: All results in one table
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter
import pandas as pd

LABELS = ['YES', 'NO', 'INSUFFICIENT_INFO']

def summarize(system, gold_labels, pred_labels):
    gold_labels, pred_labels = list(gold_labels), list(pred_labels)
    return {'system': system,
            'accuracy': round(accuracy_score(gold_labels, pred_labels), 3),
            'macro_f1': round(f1_score(gold_labels, pred_labels, labels=LABELS, average='macro', zero_division=0), 3),
            'pred_counts': dict(Counter(pred_labels))}

summary = [
    summarize('Baseline: original prompt', [r['gold'] for r in results], [r['prediction'] for r in results]),
    summarize('H2: prompt V2', [r['gold'] for r in results_v2], [r['prediction'] for r in results_v2]),
    summarize('H3: V2 + code for numbers', [r['gold'] for r in results_v3], [r['prediction'] for r in results_v3]),
    summarize('H4: logit scoring + INS bias', [r['gold'] for r in results_logit], [r['prediction'] for r in results_logit]),
    summarize('H5: LoRA SFT', main_df.gold, main_df.prediction),
]
print(f'Retriever for all systems: {RETRIEVER} top-{TOP_K}')
pd.DataFrame(summary)

Retriever for all systems: tfidf top-3


,system,accuracy,macro_f1,pred_counts
0,Baseline: original prompt,0.500,0.413,"{'NO': 20, 'YES': 4}"
1,H2: prompt V2,0.500,0.488,"{'NO': 9, 'INSUFFICIENT_INFO': 13, 'YES': 2}"
2,H3: V2 + code for numbers,0.583,0.597,"{'YES': 4, 'NO': 8, 'INSUFFICIENT_INFO': 12}"
3,H4: logit scoring + INS bias,0.542,0.550,"{'NO': 15, 'INSUFFICIENT_INFO': 5, 'YES': 4}"
4,H5: LoRA SFT,0.917,0.917,"{'YES': 8, 'NO': 8, 'INSUFFICIENT_INFO': 8}"
